# CleanGanga-Prayagraj — Data Quality & Preparation

This notebook prepares the Prayagraj river monitoring dataset
for pollution classification and persistent hotspot detection.

## Objectives
- Inspect the dataset structure
- Identify available water-quality parameters
- Check missing values
- Validate station-level observations
- Prepare the data for threshold-based pollution classification

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the Prayagraj river monitoring dataset.
# 'r' prevents Windows backslashes (\) from being interpreted as escape characters.
df = pd.read_csv(r"C:\Users\srich\OneDrive\Desktop\Technical\Projects\Internships\1M1B Virtual Internship\CleanGanga-Prayagraj\data\Prayagraj_Data.csv")

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [2]:
# Number of rows and columns
print("Shape:", df.shape)

# Names of all columns
print("\nColumns:")
print(df.columns.tolist())

# Data types and non-null counts
print("\nDataset information:")
df.info()

Shape: (1414, 23)

Columns:
['SlNo', 'Station', 'Agency', 'State LGD Code', 'State', 'District LGD Code', 'District', 'Tehsil', 'Block', 'Village', 'River', 'Basin', 'Tributary', 'Subtributary', 'SubSubtributary', 'Local River', 'Latitude', 'Longitude', 'Data Acquisition Time', 'Biochemical Oxygen Demand (mg/L)', 'Chemical Oxygen Demand (mg/L)', 'Fecal Coliform (MPN/100mL)', 'Total Coliform (MPN/100mL)']

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 1414 entries, 0 to 1413
Data columns (total 23 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   SlNo                              1414 non-null   int64  
 1   Station                           1414 non-null   str    
 2   Agency                            1414 non-null   str    
 3   State LGD Code                    1414 non-null   int64  
 4   State                             1414 non-null   str    
 5   District LGD Code           

In [3]:
# Keep only records belonging to the Prayagraj/Allahabad district.
# The dataset uses "Allahabad" as the district name.
prayagraj = df[df["District"] == "Allahabad"].copy()

print("Prayagraj records:", len(prayagraj))
print("Number of stations:", prayagraj["Station"].nunique())

print("\nObservations per station:")
print(prayagraj["Station"].value_counts())

Prayagraj records: 116
Number of stations: 6

Observations per station:
Station
GANGA AT ALLAHABAD D/S (SANGAM) U.P.           24
GANGA AT ALLAHABAD (RASOOLABAD) U.P.           24
GANGA AT KADAGHAT ALLAHABAD                    24
RIVER GANGA A/C TAMSA RIVER SIRSA SON BARSA    24
YAMUNA AT ALLAHABAD D/S (BALUA GHAT) U.P       11
TONS AT CHAKGHAT M.P.                           9
Name: count, dtype: int64


In [4]:
# Display columns from the water-quality section of the dataset.
# We previously observed that the measurement columns start around index 18.
print("Water-quality columns:")
print(df.columns[18:].tolist())

Water-quality columns:
['Data Acquisition Time', 'Biochemical Oxygen Demand (mg/L)', 'Chemical Oxygen Demand (mg/L)', 'Fecal Coliform (MPN/100mL)', 'Total Coliform (MPN/100mL)']


In [5]:
# Statistical summary of the water-quality parameters
prayagraj[df.columns[18:]].describe()

,Biochemical Oxygen Demand (mg/L),Chemical Oxygen Demand (mg/L),Fecal Coliform (MPN/100mL),Total Coliform (MPN/100mL)
count,115.000000,0.0,108.000000,0.0
mean,2.584348,NaN,862.500000,NaN
std,0.318877,NaN,294.255674,NaN
min,1.100000,NaN,2.000000,NaN
25%,2.600000,NaN,707.500000,NaN
50%,2.700000,NaN,825.000000,NaN
75%,2.800000,NaN,1100.000000,NaN
max,2.900000,NaN,1400.000000,NaN


In [ ]:
# Count missing values for each water-quality parameter
print("Missing values:")
print(prayagraj[df.columns[18:]].isna().sum())

In [ ]:
# Get all measurement columns directly from the dataframe.
measurement_columns = df.columns[18:].tolist()

print("Parameters available for analysis:")
for i, column in enumerate(measurement_columns, start=1):
    print(f"{i}. {column}")

Parameters available for analysis:
1. Data Acquisition Time
2. Biochemical Oxygen Demand (mg/L)
3. Chemical Oxygen Demand (mg/L)
4. Fecal Coliform (MPN/100mL)
5. Total Coliform (MPN/100mL)


Now we convert the data to mathematical form for further use.

In [8]:
# Measurement columns that we will actually use for pollution analysis
measurement_columns = [
    "Biochemical Oxygen Demand (mg/L)",
    "Chemical Oxygen Demand (mg/L)",
    "Fecal Coliform (MPN/100mL)",
    "Total Coliform (MPN/100mL)"
]

# Convert measurement columns to numeric values.
# If any value cannot be converted, it becomes NaN instead of causing an error.
for column in measurement_columns:
    prayagraj[column] = pd.to_numeric(
        prayagraj[column],
        errors="coerce"
    )

# Check how many missing values remain
print("Missing values after conversion:")
print(prayagraj[measurement_columns].isna().sum())

Missing values after conversion:
Biochemical Oxygen Demand (mg/L)      1
Chemical Oxygen Demand (mg/L)       116
Fecal Coliform (MPN/100mL)            8
Total Coliform (MPN/100mL)          116
dtype: int64


In [10]:

# Environmental thresholds for bathing-water assessment

# BOD:
# For bathing water, BOD should be <= 3 mg/L.
BOD_LIMIT = 3.0

# Fecal Coliform:
# 500 MPN/100mL is the desirable bathing-water limit.
FC_DESIRABLE = 500

# 2500 MPN/100mL is the maximum bathing-water limit.
FC_MAXIMUM = 2500

print("BOD limit:", BOD_LIMIT, "mg/L")
print("FC desirable limit:", FC_DESIRABLE, "MPN/100mL")
print("FC maximum limit:", FC_MAXIMUM, "MPN/100mL")

BOD limit: 3.0 mg/L
FC desirable limit: 500 MPN/100mL
FC maximum limit: 2500 MPN/100mL


Now we make the raw measurements to pollution indicators.

In [12]:
# Create pollution indicators

# True means BOD is above the acceptable bathing-water limit.
prayagraj["BOD_exceeds"] = (prayagraj["Biochemical Oxygen Demand (mg/L)"] > BOD_LIMIT)

# True means Fecal Coliform is above the desirable level.
prayagraj["FC_above_desirable"] = (prayagraj["Fecal Coliform (MPN/100mL)"] > FC_DESIRABLE)

# True means Fecal Coliform exceeds the maximum acceptable level.
prayagraj["FC_exceeds_maximum"] = (prayagraj["Fecal Coliform (MPN/100mL)"] > FC_MAXIMUM)

# Display the result
prayagraj[
    [
        "Station",
        "Biochemical Oxygen Demand (mg/L)",
        "Fecal Coliform (MPN/100mL)",
        "BOD_exceeds",
        "FC_above_desirable",
        "FC_exceeds_maximum"
    ]].head(10)

,Station,Biochemical Oxygen Demand (mg/L),Fecal Coliform (MPN/100mL),BOD_exceeds,FC_above_desirable,FC_exceeds_maximum
85,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,2.0,790.0,False,True,False
86,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,2.6,1300.0,False,True,False
87,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,2.6,930.0,False,True,False
88,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,2.5,1100.0,False,True,False
89,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,2.7,1200.0,False,True,False
90,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,2.8,1400.0,False,True,False
91,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,2.7,1300.0,False,True,False
92,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,2.7,1100.0,False,True,False
93,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,2.6,1200.0,False,True,False
94,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,2.8,1200.0,False,True,False


Classifying the pollution using multiple factors

In [ ]:
# Overall pollution classification

def classify_pollution(row):

    # Severe condition:
    # Fecal Coliform has crossed the maximum bathing-water limit.
    if row["FC_exceeds_maximum"]:
        return "High Pollution"

    # Moderate condition:
    # Either BOD exceeds its limit OR FC exceeds desirable level.
    elif row["BOD_exceeds"] or row["FC_above_desirable"]:
        return "Moderate Pollution"

    # Both indicators are within the selected limits.
    else:
        return "Low Pollution"


# Apply the classification to every observation
prayagraj["Pollution_Level"] = prayagraj.apply(classify_pollution,axis=1)
# axis=1 makes it to process one row at a function by calling the function.
# See the result
print(prayagraj["Pollution_Level"].value_counts())

Pollution_Level
Moderate Pollution    100
Low Pollution          16
Name: count, dtype: int64


In [15]:
# Show the raw measurements alongside our classification

prayagraj[
    [
        "Station",
        "Biochemical Oxygen Demand (mg/L)",
        "Fecal Coliform (MPN/100mL)",
        "BOD_exceeds",
        "FC_above_desirable",
        "FC_exceeds_maximum",
        "Pollution_Level"
    ]
].head(20)

,Station,Biochemical Oxygen Demand (mg/L),Fecal Coliform (MPN/100mL),BOD_exceeds,FC_above_desirable,FC_exceeds_maximum,Pollution_Level
85,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,2.0,790.0,False,True,False,Moderate Pollution
86,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,2.6,1300.0,False,True,False,Moderate Pollution
87,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,2.6,930.0,False,True,False,Moderate Pollution
88,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,2.5,1100.0,False,True,False,Moderate Pollution
89,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,2.7,1200.0,False,True,False,Moderate Pollution
90,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,2.8,1400.0,False,True,False,Moderate Pollution
91,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,2.7,1300.0,False,True,False,Moderate Pollution
92,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,2.7,1100.0,False,True,False,Moderate Pollution
93,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,2.6,1200.0,False,True,False,Moderate Pollution
94,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,2.8,1200.0,False,True,False,Moderate Pollution
